# idtracker.ai + your Detectron2 weights (inference only)

For a project that already has **trained Detectron2 weights** and a **`roi.json`**.
No annotation, dataset or training: this notebook runs the model over your videos on
the Colab GPU, writes one contour file per clip, then tracks.

Put these on Drive (folder name is set in the config cell):

```
idtrackerai_detectron2/
  model/model_final.pth      your weights  (training_metadata.json beside it if you have it)
  roi.json                   your region of interest
  clips/*.mp4                the videos
```
Runtime -> Change runtime type -> GPU first.

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "No GPU: switch the runtime to GPU and rerun.")
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

# ---- the only cell you should need to edit ---------------------------------
PROJECT    = Path('/content/drive/MyDrive/idtrackerai_detectron2')
VIDEOS     = PROJECT / 'clips'
VIDEO_GLOB = '*.mp4'
WEIGHTS    = PROJECT / 'model' / 'model_final.pth'
ROI_JSON   = PROJECT / 'roi.json'

N_ANIMALS       = 5
SCORE_THRESHOLD = 0.7
ON_OVERLAP      = 'merge'   # 'merge' keeps crossings for idtracker.ai to resolve

# Only used if there is NO training_metadata.json next to the weights.
# Must match what the weights were fine-tuned from.
MODEL_CONFIG = 'COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml'
NUM_CLASSES  = 1
MIN_SIZE     = 640          # inference scale used in training
# -----------------------------------------------------------------------------

CONTOURS = PROJECT / 'contours'
SESSIONS = PROJECT / 'sessions'
WHEELS   = PROJECT / 'wheels'
CACHE    = Path('/content/cache')
for d in (CONTOURS, SESSIONS, WHEELS):
    d.mkdir(parents=True, exist_ok=True)

videos = sorted(VIDEOS.glob(VIDEO_GLOB))
for label, p in [('weights', WEIGHTS), ('roi.json', ROI_JSON), ('videos', VIDEOS)]:
    print(f"{label:9s}{'OK     ' if p.exists() else 'MISSING'} {p}")
print(f"{len(videos)} video(s) match {VIDEO_GLOB}")
assert WEIGHTS.is_file() and ROI_JSON.is_file() and videos, 'fix the paths above'
HAS_META = (WEIGHTS.parent / 'training_metadata.json').is_file()
print('training_metadata.json:', 'found' if HAS_META else 'absent -> MODEL_CONFIG/NUM_CLASSES/MIN_SIZE are used')

## Install Detectron2 (cached to Drive) and this fork

In [ ]:
import subprocess, sys, torch
key = f"py{sys.version_info.major}{sys.version_info.minor}-torch{torch.__version__}"
cache = WHEELS / key
cache.mkdir(parents=True, exist_ok=True)

def run(cmd):
    print('$', ' '.join(cmd), flush=True)
    return subprocess.run(cmd).returncode

try:
    import detectron2
    print('detectron2 already importable')
except ImportError:
    wheels = list(cache.glob('detectron2*.whl'))
    if not wheels:
        print('Building detectron2 (~10 min), cached afterwards')
        run([sys.executable, '-m', 'pip', 'wheel', '--no-deps', '--no-build-isolation', '-q',
             '--wheel-dir', str(cache), 'git+https://github.com/facebookresearch/detectron2.git'])
        wheels = list(cache.glob('detectron2*.whl'))
    assert wheels, 'detectron2 build failed; see log'
    run([sys.executable, '-m', 'pip', 'install', '-q', str(wheels[0])])
run([sys.executable, '-m', 'pip', 'install', '-q', 'git+https://github.com/mbel-tech/idtrackerai-detectron2.git'])
import torch, detectron2
assert torch.cuda.is_available(), 'No CUDA: switch runtime to GPU'
print('detectron2', detectron2.__version__, '| torch', torch.__version__, '|', torch.cuda.get_device_name(0))

## Convert `roi.json` to idtracker.ai's `roi_list`

idtracker.ai stores each region as a string such as `+ Polygon [[x, y], ...]`
(`+` adds, `-` removes). This cell accepts several common layouts of `roi.json`:
already-converted strings, a list of polygons, `{"polygons": ...}` / `{"points": ...}`,
or LabelMe-style `{"shapes": [...]}`. **Check the overlay it draws.** If your file is
laid out differently the cell stops and prints the top of the file instead of guessing.

In [ ]:
import json, cv2, numpy as np
from IPython.display import display
from PIL import Image

raw = json.loads(ROI_JSON.read_text(encoding='utf-8'))

def is_poly(x):
    return (isinstance(x, list) and len(x) >= 3 and
            all(isinstance(p, (list, tuple)) and len(p) == 2 and
                all(isinstance(v, (int, float)) for v in p) for p in x))

def fmt(sign, pts):
    return sign + ' Polygon ' + json.dumps([[round(float(x), 1), round(float(y), 1)] for x, y in pts])

def to_roi_list(obj):
    if isinstance(obj, str):
        return [obj]
    if isinstance(obj, list) and obj and all(isinstance(s, str) for s in obj):
        return obj                                   # already idtracker strings
    if isinstance(obj, dict):
        for k in ('roi_list', 'polygons', 'rois', 'roi', 'shapes', 'regions'):
            if k in obj:
                return to_roi_list(obj[k])
        if 'points' in obj:
            return to_roi_list(obj['points'])
    if is_poly(obj):
        return [fmt('+', obj)]
    if isinstance(obj, list):
        out = []
        for item in obj:
            if isinstance(item, dict) and is_poly(item.get('points')):
                mode = str(item.get('mode', item.get('label', ''))).lower()
                out.append(fmt('-' if mode in ('-', 'exclude', 'remove') else '+', item['points']))
            else:
                out += to_roi_list(item)
        return out
    raise ValueError('unrecognised roi.json layout')

try:
    ROI_LIST = to_roi_list(raw)
    assert ROI_LIST
except Exception as e:
    print('Could not interpret roi.json:', e)
    print('Top of the file:', json.dumps(raw)[:600])
    raise SystemExit

print(len(ROI_LIST), 'region(s)')
for r in ROI_LIST:
    print('  ', r[:90])

# overlay on the first frame of the first video
cap = cv2.VideoCapture(str(videos[0])); ok, frame = cap.read(); cap.release()
from idtrackerai.utils.py_utils import build_ROI_mask_from_list
mask = build_ROI_mask_from_list(ROI_LIST, frame.shape[1], frame.shape[0])
shade = frame.copy()
shade[mask == 0] = (shade[mask == 0] * 0.3).astype(np.uint8)
display(Image.fromarray(cv2.cvtColor(shade, cv2.COLOR_BGR2RGB)).resize((800, int(800 * frame.shape[0] / frame.shape[1]))))
print('Bright area = tracked. Dark area = ignored.')

## Image enhancement (CLAHE and light edits)
The model must see the same kind of image at inference as it was trained on, and idtracker.ai cuts
its identification images from the same enhanced frame, so **one set of settings is used for export
and tracking** and is recorded in full everywhere it is used.

Resolution order, later wins: built-in defaults -> `training_metadata.json` beside the weights ->
`PROJECT/preprocess_profile.json` (if present) -> `ENHANCEMENT_OVERRIDES` below.
Keys: `enhance, clahe_clip, clahe_tile, correct_lighting, illumination_sigma, illumination_downsample,
exposure, brightness, contrast, gamma, shadows, highlights, blacks, whites, sharpness, denoise`.

In [ ]:
from types import SimpleNamespace
from idtrackerai.extra_tools.detectron2_pipeline import preprocessing as fp

# Anything you want to force, e.g. {'clahe_clip': 2.0, 'clahe_tile': 8}. Leave empty to use the above.
ENHANCEMENT_OVERRIDES = {}

meta_path = WEIGHTS.parent / 'training_metadata.json'
trained_with = json.loads(meta_path.read_text()).get('enhancement') if meta_path.is_file() else None
profile = PROJECT / 'preprocess_profile.json'
ns = SimpleNamespace(preprocess_profile=profile if profile.is_file() else None)
settings, notes = fp.resolve_settings(ns, fallback=trained_with, fallback_label='the training record')
settings = fp.normalize_settings({**settings, **ENHANCEMENT_OVERRIDES})

ENHANCEMENT = settings
ENHANCEMENT_SOURCES = notes + ([f'overrides: {ENHANCEMENT_OVERRIDES}'] if ENHANCEMENT_OVERRIDES else [])
ENH_PROFILE = PROJECT / 'enhancement_used.json'
fp.save_profile(ENH_PROFILE, ENHANCEMENT, name='used by this run', notes='; '.join(ENHANCEMENT_SOURCES))

print(fp.describe(ENHANCEMENT))
print(json.dumps(ENHANCEMENT, indent=2))
print('sources:', *ENHANCEMENT_SOURCES, sep='\n  ')
if trained_with:
    msg = fp.check_settings_match(trained_with, ENHANCEMENT, 'training')
    print('\nWARNING:' if msg else '\nMatches the training record.', msg or '')
else:
    print('\nNo training record: make sure these match how the model was trained.')

## Export contours (GPU, resumable)
Each clip writes `contours/<clip>.h5`, with the enhancement, score threshold, overlap mode,
post-processing and weights hash stored inside it. After a disconnect, rerun this cell: finished clips are skipped.

In [ ]:
extra = '' if HAS_META else f'--config {MODEL_CONFIG} --num-classes {NUM_CLASSES} --min-size {MIN_SIZE}'
!idtrackerai_d2_export --videos {VIDEOS}/{VIDEO_GLOB} --weights {WEIGHTS} --output-dir {CONTOURS} \
    --local-cache {CACHE} --max-instances {N_ANIMALS} --score-threshold {SCORE_THRESHOLD} \
    --on-overlap {ON_OVERLAP} --preprocess-profile {ENH_PROFILE} --progress-every 1000 {extra}

## Parameter files (ROI included), one per recording

In [ ]:
import json, math, tomllib
from idtrackerai.extra_tools.detectron2_pipeline import dataset as ds

def tv(v):
    if isinstance(v, bool):
        return 'true' if v else 'false'
    if isinstance(v, float) and math.isinf(v):
        return 'inf' if v > 0 else '-inf'
    if isinstance(v, (list, tuple)):
        return '[' + ', '.join(tv(x) for x in v) + ']'
    if isinstance(v, dict):
        return '{' + ', '.join(f'{k} = {tv(x)}' for k, x in v.items()) + '}'
    return json.dumps(v)

# The full enhancement dict goes into every session file, so the tracking run (which cuts its
# identification images from the enhanced frame) uses exactly what the export used, and
# session.json records it.
BASE = {'number_of_animals': N_ANIMALS, 'roi_list': ROI_LIST}
# Optional: a parameters .toml from the Segmentation App in PROJECT supplies area_ths etc.
tomls = sorted(PROJECT.glob('*.toml'))
if tomls:
    BASE = {**tomllib.loads(tomls[0].read_text(encoding='utf-8')),
            'roi_list': ROI_LIST, 'number_of_animals': N_ANIMALS}
    print('Inheriting', tomls[0].name, '(roi_list from roi.json overrides it)')
    if BASE.get('enhancement') and BASE['enhancement'] != ENHANCEMENT:
        print('NOTE: the inherited enhancement differs; the one used for export replaces it.')
BASE['enhancement'] = ENHANCEMENT
for k in ('video_paths', 'external_contours', 'name', 'session'):
    BASE.pop(k, None)

groups = ds.group_videos([v.stem for v in videos])
for name, members in sorted(groups.items()):
    clips = [v for v in videos if v.stem in members]
    h5s = [CONTOURS / f'{c.stem}.h5' for c in clips]
    if any(not h.is_file() for h in h5s):
        print('skip', name, '- contours missing')
        continue
    params = {**BASE, 'video_paths': [str(c) for c in clips],
              'external_contours': [str(h) for h in h5s], 'name': name}
    out = SESSIONS / f'{name}.toml'
    out.write_text('\n'.join(f'{k} = {tv(v)}' for k, v in params.items()) + '\n', encoding='utf-8')
    print('wrote', out)

    # Provenance beside the session: everything that shaped these contours and this run.
    import hashlib, datetime
    prov = {
        'written': datetime.datetime.now().isoformat(timespec='seconds'),
        'session': name,
        'videos': [str(c) for c in clips],
        'contours': [str(h) for h in h5s],
        'weights': str(WEIGHTS),
        'weights_sha256': hashlib.sha256(WEIGHTS.read_bytes()).hexdigest(),
        'model': {'config': 'training_metadata.json' if HAS_META else MODEL_CONFIG,
                  'num_classes': None if HAS_META else NUM_CLASSES,
                  'min_size': None if HAS_META else MIN_SIZE},
        'training_metadata': json.loads(meta_path.read_text()) if HAS_META else None,
        'export': {'score_threshold': SCORE_THRESHOLD, 'max_instances': N_ANIMALS, 'on_overlap': ON_OVERLAP},
        'enhancement': ENHANCEMENT,
        'enhancement_description': fp.describe(ENHANCEMENT),
        'enhancement_sources': ENHANCEMENT_SOURCES,
        'trained_with_enhancement': trained_with,
        'roi_source': str(ROI_JSON),
        'roi_list': ROI_LIST,
        'number_of_animals': N_ANIMALS,
    }
    (SESSIONS / f'{name}.provenance.json').write_text(json.dumps(prov, indent=2), encoding='utf-8')

## Track
Segmentation is already done (contours come from file; the ROI is applied to them by centroid),
so this only runs idtracker.ai's identification. **Not verified on a Colab runtime.**
If it fails, download `contours/` and `sessions/` and run `idtrackerai --parameters <file>.toml --track` locally.

In [ ]:
import os, subprocess
os.environ['QT_QPA_PLATFORM'] = 'offscreen'
for path in sorted(SESSIONS.glob('*.toml')):
    print('===', path.stem, flush=True)
    print('exit', subprocess.run(['idtrackerai', '--parameters', str(path), '--track']).returncode)